# **Mount Drive**

# **Load Data**

In [ ]:
import pandas as pd
ffail = '<DATA_ROOT>/crop_failure_modified/crops_failure_0595_20240221.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])

cond1 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond1]
df_fail.loc[:,'fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]

cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']

df_fail = df_fail[cols]
df_fail.groupby('Irrigation Practice').count()

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
  detrended_yield = np.zeros(len(yield_series))
  if len(yield_series) > 2:
    detrended_yield = detrend(yield_series)
  return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)
dfy['d_yield_standard'] = dfy['detrended_yield'] / (dfy['value_yield'] - dfy['detrended_yield'])

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])

cols = ['FIPS','commodity_desc','prodn_practice_desc','year','d_yield_standard']
df_yield = dfy[cols]
condy = df_yield.year > 2008
df_yield = df_yield[condy]
# df_yield
df_yield.groupby('prodn_practice_desc').count()

In [ ]:
from glob import glob
import pandas as pd
in_path = '<DATA_ROOT>/SVI_weightedCrop_Final/*.csv'
files = glob(in_path)

list_df = []
for f in files:
  crop = f.split('/')[-1].split('.')[0][4:]
  # print(crop)
  dfi = pd.read_csv(f).copy()
  dfi['EPL_PCI'] = dfi['EPL_PCI'].fillna(dfi['EPL_HBURD'])
  dfi = dfi.drop(columns=['EPL_HBURD' ,	'EPL_UNINSUR'])
  dfi['Crop'] = crop
  list_df.append(dfi)

df_svi = pd.concat(list_df,ignore_index=True)
df_svi = df_svi.set_index(['STCNTY','YEAR','Crop']).dropna(how='all').reset_index()
df_svi = df_svi.rename(columns={'STCNTY':'FIPS','YEAR':'year'})
df_svi

In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate']]
dfw['FIPS'] = dfw['FIPS'].astype(int)
dfw

In [ ]:
finsure = '<DATA_ROOT>/InsuranceData/InsuranceData.csv'
df_insure = pd.read_csv(finsure)
df_insure['FIPS'] = df_insure['State_Code'] * 1000 + df_insure['County_Code']
df_insure

In [ ]:
df_insure.columns

In [ ]:
finsure = '<DATA_ROOT>/InsuranceData/InsuranceData.csv'
df_insure = pd.read_csv(finsure)
df_insure['FIPS'] = df_insure['State_Code'] * 1000 + df_insure['County_Code']
cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc']
# cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc','Subsidy','Liability']
df_insure = df_insure[cols]
df_insure = df_insure.rename(columns={'Commodity_Name':'Crop','Year_Identifier':'year'})
cond1 = df_insure['Cause_of_Loss_Code'] != 'XX'
df_insure = df_insure[cond1]
df_insure['Cause_of_Loss_Code'] = df_insure['Cause_of_Loss_Code'].astype(int)
df_insure = df_insure.groupby(['FIPS','Crop','year','Cause_of_Loss_Code']).first().reset_index()
df_insure

In [9]:
fcold = '<DATA_ROOT>/ClimateIndex/ColdSpell.csv'
df_cold = pd.read_csv(fcold)
cols = ['FIPS','Year','Thr-2','Thr-1','Thr0']
df_cold = df_cold[cols]
df_cold = df_cold.dropna(subset=['FIPS', 'Year'])
df_cold['FIPS'] = df_cold['FIPS'].astype(int)
df_cold['Year'] = df_cold['Year'].astype(int)
df_cold = df_cold.rename(columns={'Year':'year'})

In [10]:
fdrought = '<DATA_ROOT>/ClimateIndex/list_droughtFeatures.csv'
df_drought = pd.read_csv(fdrought)
df_drought = df_drought.dropna(subset=['Fips', 'Year'])
df_drought['Fips'] = df_drought['Fips'].astype(int)
df_drought['Year'] = df_drought['Year'].astype(int)
df_drought = df_drought.rename(columns={'Fips':'FIPS'})
df_drought = df_drought.rename(columns={'Year':'year'})

In [11]:
fprcp = '<DATA_ROOT>/ClimateIndex/prcp.csv'
df_prcp = pd.read_csv(fprcp)
df_prcp = df_prcp.dropna(subset=['FIPS', 'Year'])
df_prcp['FIPS'] = df_prcp['FIPS'].astype(int)
df_prcp['Year'] = df_prcp['Year'].astype(int)
df_prcp = df_prcp.drop(columns=['State','County'])
df_prcp = df_prcp.rename(columns={'Year':'year'})

In [12]:
fvpd = '<DATA_ROOT>/ClimateIndex/vpdfeatures.csv'
df_vpd = pd.read_csv(fvpd)
df_vpd = df_vpd.dropna(subset=['FIPS', 'Year'])
df_vpd['FIPS'] = df_vpd['FIPS'].astype(int)
df_vpd['Year'] = df_vpd['Year'].astype(int)
df_vpd = df_vpd.drop(columns=['State','County'])
df_vpd = df_vpd.rename(columns={'Year':'year'})

In [13]:
fwarm = '<DATA_ROOT>/ClimateIndex/WarmSpell.csv'
df_warm = pd.read_csv(fwarm)
df_warm = df_warm.dropna(subset=['FIPS', 'Year'])
df_warm['FIPS'] = df_warm['FIPS'].astype(int)
df_warm['Year'] = df_warm['Year'].astype(int)
df_warm = df_warm.drop(columns=['State','County'])
df_warm = df_warm.rename(columns={'Year':'year'})

In [14]:
# df_fail , df_yield , df_svi , dfw , df_insure

In [ ]:
df_insure = df_insure.drop(columns=['Cause_of_Loss_Code'])
df_insure

In [16]:
df_svi = df_svi[['FIPS','Crop','year','RPL_THEMES']]
df_cold = df_cold[['FIPS', 'year','Thr-2']]
df_drought = df_drought[['FIPS', 'year','pdsi-2']] # 'spi1y-2'
df_prcp = df_prcp[['FIPS', 'year','dpi1']]
df_vpd = df_vpd[['FIPS', 'year','VPD-Mean']]
df_warm = df_warm[['FIPS', 'year','Thr31']]


# **Yield Analysis**

In [ ]:
cond_irig = df_yield['prodn_practice_desc'] == 'ALL PRODUCTION PRACTICES'
df_yield = df_yield[cond_irig]
df_yield['prodn_practice_desc'] = 'ALL'
df_yield = df_yield.rename(columns={'commodity_desc':'Crop','prodn_practice_desc':'Irrigation Practice'})
df_yield['FIPS'] = df_yield['FIPS'].astype(int)

In [34]:
df_yield.iloc[0:3]

,FIPS,Crop,Irrigation Practice,year,d_yield_standard
42,4003,BARLEY,ALL,2009,-0.026138
231,4003,HAY,ALL,2009,0.018419
232,4003,HAY,ALL,2010,-0.099102


In [ ]:
df_merge_yield_ins = pd.merge(df_yield,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_yield_ins_svi = pd.merge(df_merge_yield_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_yield_ins_svi_gwu = pd.merge(df_merge_yield_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')
# df_merge_crop_ins_svi_gwu

df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu.dropna(subset='d_yield_standard')
condy = df_merge_yield_ins_svi_gwu.year < 2023
df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu[condy]

# df_merge_crop_ins_svi_gwu['Liability'] = df_merge_crop_ins_svi_gwu['Liability'].fillna(0)
# df_merge_crop_ins_svi_gwu['Subsidy'] = df_merge_crop_ins_svi_gwu['Subsidy'].fillna(0)
# df_merge_crop_ins_svi_gwu

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])
# df_merge_clim

df_merge = pd.merge(df_merge_yield_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()
df_merge

In [36]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]

df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]

In [37]:
Cause_of_Loss[-3]

'Drought'

In [ ]:
# cond_ir = df_merge['Irrigation Practice'] == 'ALL'
# df_merge_yield = df_merge[cond_ir]
# df_merge_yield.to_csv('<DATA_ROOT>/Final_Exports_2024_2/yield_vars.csv')

In [38]:
cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond_ir = df_merge['Irrigation Practice'] == 'ALL'
df_merge_yield = df_merge[cond_ir & cond_loss]

In [ ]:
df_merge_yield.groupby('Crop').count()

In [49]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'SOYBEANS' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_yield.Crop == selected_crop
df = df_merge_yield[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# df_scaled

In [50]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=49) #

params = {'max_depth': 10,
          'learning_rate': 0.01,
          'subsample': 0.6,
          'n_estimators': 750,
          }

xgb_model = xgb.XGBRegressor(**params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)


R-squared_test: 0.4502600637230836
R-squared_train: 0.8869461958822178


In [ ]:
# params = {'max_depth': 7,
#           'learning_rate': 0.001,
#           'subsample': 0.8,
#           'colsample_bytree': 0.75,
#           'colsample_bylevel': 0.75,
#           'n_estimators': 100,
#           'max_leaves':100
#             }

In [ ]:
learning_rate = [i/1000. for i in range(8,14)]
learning_rate

[0.008, 0.009, 0.01, 0.011, 0.012, 0.013]

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)


# mean_train = np.mean(y_train)
# # Get predictions on the test set
# baseline_predictions = np.ones(y_test.shape) * mean_train
# # Compute MAE
# mae_baseline = mean_absolute_error(y_test, baseline_predictions)
# print("Baseline MAE is {:.2f}".format(mae_baseline))

params = {
      'max_depth': 6,
      'n_estimators': 750,
      'subsample': 0.6,
}

# learning_rate = [i/1000. for i in range(8,14)]


params['eval_metric'] = "mae"
num_boost_round = 999

gridsearch_params = [ learning_rate  for learning_rate in [i/1000. for i in range(8,14)]]
# gridsearch_params = [
#     (max_depth, min_child_weight,subsample, colsample,eta)
#     for max_depth in range(9,12)
#     for min_child_weight in range(5,8)
#     for subsample in [i/10. for i in range(7,11)]
#     for colsample in [i/10. for i in range(7,11)]
#     for eta in [.3, .2, .1, .05, .01, .005]
# ]

min_mae = float("Inf")
best_params = None
# for max_depth, min_child_weight,subsample, colsample,eta in gridsearch_params:
for learning_rate in gridsearch_params:
    # print("max_depth={}, min_child_weight={}, subsample={}, colsample={}, eta={}".format(
    #     max_depth,min_child_weight,subsample, colsample,eta))
    print(learning_rate)
    # Update our parameters
    # params['max_depth'] = max_depth
    # params['min_child_weight'] = min_child_weight
    # params['subsample'] = subsample
    # params['colsample_bytree'] = colsample
    params['learning_rate'] = learning_rate

    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=10,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best MAE
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {}".format(mean_mae))
    if mean_mae < min_mae:
      min_mae = mean_mae
      # best_params = (max_depth,min_child_weight,subsample, colsample,eta)
      best_params = (learning_rate)

print("Best params: {}, MAE: {}".format(best_params, min_mae))

# params['max_depth'] = best_params[0]
# params['min_child_weight'] = best_params[1]
# params['subsample'] = best_params[2]
# params['colsample_bytree'] = best_params[3]
params['learning_rate'] = best_params

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=50
)
print("Best MAE: {:.2f} in {} rounds".format(model.best_score, model.best_iteration+1))

num_boost_round = model.best_iteration + 1
best_model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

best_model.save_model("my_model.model")

y_pred = best_model.predict(dtest)
y_pred2 = best_model.predict(dtrain)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)

In [ ]:
best_params

0.013

In [ ]:
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33,random_state=42)

from sklearn.svm import SVR

svr_rbf = SVR(kernel='rbf',gamma='scale', C=1.0, epsilon=0.1)
svr_rbf.fit(X_train, y_train)

print(svr_rbf.score(X_test,y_test))

from sklearn.metrics import mean_squared_error

print("RMSE for RBF kernelized SVR:",np.sqrt(mean_squared_error(y_test,svr_rbf.predict(X_test))))

from sklearn.metrics import r2_score
y_pred = svr_rbf.predict(X_test)
y_pred2 = svr_rbf.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)



0.20091606250253602
RMSE for RBF kernelized SVR: 0.9131131401725503


In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'C':[0.01,0.05,0.1,0.5,1,2,5],'epsilon':[0.1,0.2,0.5,1]}

grid = GridSearchCV(svr_rbf,param_grid=params,cv=5,scoring='r2',verbose=1,return_train_score=True)

grid.fit(X_train,y_train)



Fitting 5 folds for each of 28 candidates, totalling 140 fits


GridSearchCV(cv=5, estimator=SVR(),
             param_grid={'C': [0.01, 0.05, 0.1, 0.5, 1, 2, 5],
                         'epsilon': [0.1, 0.2, 0.5, 1]},
             return_train_score=True, scoring='r2', verbose=1)

In [ ]:
grid.best_estimator_

SVR(C=5, epsilon=0.5)

In [ ]:

svr_best=SVR(kernel='rbf',gamma='scale', C=5.0, epsilon=0.5)
svr_best.fit(X_train, y_train)

print("RMSE for RBF kernelized SVR:",np.sqrt(mean_squared_error(y_test,svr_rbf.predict(X_test))))

from sklearn.metrics import r2_score
y_pred = svr_rbf.predict(X_test)
y_pred2 = svr_rbf.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)

RMSE for RBF kernelized SVR: 0.9131131401725503
R-squared_test: 0.20091606250253602
R-squared_train: 0.24299916300068491


In [ ]:
X_test

# **RF tuning**

In [ ]:
cond_irig = df_yield['prodn_practice_desc'] == 'ALL PRODUCTION PRACTICES'
df_yield = df_yield[cond_irig]
df_yield['prodn_practice_desc'] = 'ALL'
df_yield = df_yield.rename(columns={'commodity_desc':'Crop','prodn_practice_desc':'Irrigation Practice'})
df_yield['FIPS'] = df_yield['FIPS'].astype(int)

In [ ]:
df_merge_yield_ins = pd.merge(df_yield,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_yield_ins_svi = pd.merge(df_merge_yield_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_yield_ins_svi_gwu = pd.merge(df_merge_yield_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')

df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu.dropna(subset='d_yield_standard')
condy = df_merge_yield_ins_svi_gwu.year < 2023
df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu[condy]

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])

df_merge = pd.merge(df_merge_yield_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()
# df_merge

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]

df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]

cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond_ir = df_merge['Irrigation Practice'] == 'ALL'
df_merge_yield = df_merge[cond_ir & cond_loss]

In [ ]:
# Exports
selected_crop = 'CORN' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_yield.Crop == selected_crop
df = df_merge_yield[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
outpath = '<DATA_ROOT>/ML_Tuning_Exports/Yield/'
df.to_csv(outpath + 'Drought_' + selected_crop + '.csv',index=False)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'WHEAT' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_yield.Crop == selected_crop
df = df_merge_yield[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# df_scaled

In [ ]:
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

features = df_scaled.drop(columns=['d_yield_standard'])
target = df_scaled['d_yield_standard']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33,random_state=42)


In [ ]:

# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error

############################################################################### Tuning Model

rf_model = RandomForestRegressor(random_state=42)
param_grid = {
'n_estimators': [120],  # The number of trees in the forest
'max_features': np.arange(6 , 9),  # mtry values from 1 to 25   #### The number of features
'min_samples_leaf': [1],#[5, 10, 15] The minimum number of samples required to be at a leaf node
'max_depth': [11],
 }
# Create the KFold cross-validator with 10 folds
kf = KFold(n_splits=10, shuffle=True, random_state=42)   ## The number of folds

# Create the GridSearchCV object
grid_search = GridSearchCV(rf_model, param_grid, scoring='neg_mean_squared_error', cv=kf,n_jobs=-1)

grid_search.fit(features, target)
n_est = grid_search.best_params_['n_estimators']
n_lef = grid_search.best_params_['min_samples_leaf']
n_mxf = grid_search.best_params_['max_features']
n_mxd = grid_search.best_params_['max_depth']

############################################################################### Loop, train, test, and Shapley Model

X_train, X_test, y_train, y_test = [] , [] , [] , []
r2_train , r2_test = [] , []
df_importance = pd.DataFrame(columns=features.columns)

array_shape = (features.shape[0], features.shape[1], 10) ## Loop
shap_arr = np.empty(array_shape)
# import shap
for i in range(10):  ## Loop
    if i % 10 == 0: print(i)
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20)
    reg = RandomForestRegressor(n_estimators=n_est, min_samples_leaf=n_lef,
                                max_features=n_mxf,max_depth=n_mxd)
    reg.fit(X_train, y_train)

    y_pred = reg.predict(X_train)
    r2 = r2_score(y_train, y_pred)
    r2_train.append(r2)
    # print("TRAIN")
    # print(f"R-squared (R2) Score: {r2}")

    y_pred = reg.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_test.append(r2)
    # print("TEST")
    # print(f"R-squared (R2) Score: {r2}")

    feature_importances = reg.feature_importances_
    df_importance.loc[i] = feature_importances
    # explainer = shap.Explainer(reg)
    # shap_values = explainer(features)
    # shap_arr[:,:,i] = shap_values.values

r2 = np.mean(r2_train)
print("TRAIN")
print(f"R-squared (R2) Score: {r2}")

r2 = np.mean(r2_test)
print("TEST")
print(f"R-squared (R2) Score: {r2}")

print(grid_search.best_params_)


0
TRAIN
R-squared (R2) Score: 0.7093356158681101
TEST
R-squared (R2) Score: 0.369170838840861
{'max_depth': 11, 'max_features': 8, 'min_samples_leaf': 1, 'n_estimators': 120}


In [ ]:
# -3
# TRAIN
# R-squared (R2) Score: 0.7333852425326542
# TEST
# R-squared (R2) Score: 0.44490459320583736
# {'max_features': 6, 'min_samples_leaf': 5, 'n_estimators': 150}

# TRAIN
# R-squared (R2) Score: 0.7735333588184085
# TEST
# R-squared (R2) Score: 0.43648050099021696
# {'max_features': 6, 'min_samples_leaf': 4, 'n_estimators': 200}

# TRAIN
# R-squared (R2) Score: 0.9234981105308148
# TEST
# R-squared (R2) Score: 0.4574035466108996
# {'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 220}

# TRAIN
# R-squared (R2) Score: 0.9233099147433868
# TEST
# R-squared (R2) Score: 0.4589719877954949
# {'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 220}

In [ ]:
# -1
# TRAIN
# R-squared (R2) Score: 0.9236805392171996
# TEST
# R-squared (R2) Score: 0.44362996097710716
# {'max_features': 5, 'min_samples_leaf': 1, 'n_estimators': 240}

# TRAIN
# R-squared (R2) Score: 0.9244202453128091
# TEST
# R-squared (R2) Score: 0.43769460560935575
# {'max_features': 5, 'min_samples_leaf': 1, 'n_estimators': 250}

# **Load Data**

In [ ]:
import pandas as pd
ffail = '<DATA_ROOT>/crop_failure_modified/crops_failure_0595_20240221.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])

cond1 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond1]
df_fail.loc[:,'fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]

cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']

df_fail = df_fail[cols]
df_fail.groupby('Irrigation Practice').count()

In [ ]:
df_fail

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
  detrended_yield = np.zeros(len(yield_series))
  if len(yield_series) > 2:
    detrended_yield = detrend(yield_series)
  return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)
dfy['d_yield_standard'] = dfy['detrended_yield'] / (dfy['value_yield'] - dfy['detrended_yield'])

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])

cols = ['FIPS','commodity_desc','prodn_practice_desc','year','d_yield_standard']
df_yield = dfy[cols]
condy = df_yield.year > 2008
df_yield = df_yield[condy]
# df_yield
df_yield.groupby('prodn_practice_desc').count()

In [ ]:
df_yield

In [ ]:
from glob import glob
import pandas as pd
in_path = '<DATA_ROOT>/SVI_weightedCrop_Final/*.csv'
files = glob(in_path)

list_df = []
for f in files:
  crop = f.split('/')[-1].split('.')[0][4:]
  # print(crop)
  dfi = pd.read_csv(f).copy()
  dfi['EPL_PCI'] = dfi['EPL_PCI'].fillna(dfi['EPL_HBURD'])
  dfi = dfi.drop(columns=['EPL_HBURD' ,	'EPL_UNINSUR'])
  dfi['Crop'] = crop
  list_df.append(dfi)

df_svi = pd.concat(list_df,ignore_index=True)
df_svi = df_svi.set_index(['STCNTY','YEAR','Crop']).dropna(how='all').reset_index()
df_svi = df_svi.rename(columns={'STCNTY':'FIPS','YEAR':'year'})
df_svi

In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate']]
dfw['FIPS'] = dfw['FIPS'].astype(int)
dfw

In [ ]:
finsure = '<DATA_ROOT>/InsuranceData/InsuranceData.csv'
df_insure = pd.read_csv(finsure)
df_insure['FIPS'] = df_insure['State_Code'] * 1000 + df_insure['County_Code']
cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc']
df_insure = df_insure[cols]
df_insure = df_insure.rename(columns={'Commodity_Name':'Crop','Year_Identifier':'year'})
cond1 = df_insure['Cause_of_Loss_Code'] != 'XX'
df_insure = df_insure[cond1]
df_insure['Cause_of_Loss_Code'] = df_insure['Cause_of_Loss_Code'].astype(int)
df_insure = df_insure.groupby(['FIPS','Crop','year','Cause_of_Loss_Code']).first().reset_index()
df_insure

In [ ]:
fcold = '<DATA_ROOT>/ClimateIndex/ColdSpell.csv'
df_cold = pd.read_csv(fcold)
cols = ['FIPS','Year','Thr-2','Thr-1','Thr0']
df_cold = df_cold[cols]
df_cold = df_cold.dropna(subset=['FIPS', 'Year'])
df_cold['FIPS'] = df_cold['FIPS'].astype(int)
df_cold['Year'] = df_cold['Year'].astype(int)
df_cold = df_cold.rename(columns={'Year':'year'})

In [ ]:
fdrought = '<DATA_ROOT>/ClimateIndex/list_droughtFeatures.csv'
df_drought = pd.read_csv(fdrought)
df_drought = df_drought.dropna(subset=['Fips', 'Year'])
df_drought['Fips'] = df_drought['Fips'].astype(int)
df_drought['Year'] = df_drought['Year'].astype(int)
df_drought = df_drought.rename(columns={'Fips':'FIPS'})
df_drought = df_drought.rename(columns={'Year':'year'})

In [ ]:
fprcp = '<DATA_ROOT>/ClimateIndex/prcp.csv'
df_prcp = pd.read_csv(fprcp)
df_prcp = df_prcp.dropna(subset=['FIPS', 'Year'])
df_prcp['FIPS'] = df_prcp['FIPS'].astype(int)
df_prcp['Year'] = df_prcp['Year'].astype(int)
df_prcp = df_prcp.drop(columns=['State','County'])
df_prcp = df_prcp.rename(columns={'Year':'year'})

In [ ]:
fvpd = '<DATA_ROOT>/ClimateIndex/vpdfeatures.csv'
df_vpd = pd.read_csv(fvpd)
df_vpd = df_vpd.dropna(subset=['FIPS', 'Year'])
df_vpd['FIPS'] = df_vpd['FIPS'].astype(int)
df_vpd['Year'] = df_vpd['Year'].astype(int)
df_vpd = df_vpd.drop(columns=['State','County'])
df_vpd = df_vpd.rename(columns={'Year':'year'})

In [ ]:
fwarm = '<DATA_ROOT>/ClimateIndex/WarmSpell.csv'
df_warm = pd.read_csv(fwarm)
df_warm = df_warm.dropna(subset=['FIPS', 'Year'])
df_warm['FIPS'] = df_warm['FIPS'].astype(int)
df_warm['Year'] = df_warm['Year'].astype(int)
df_warm = df_warm.drop(columns=['State','County'])
df_warm = df_warm.rename(columns={'Year':'year'})

In [ ]:
# df_fail , df_yield , df_svi , dfw , df_insure

In [ ]:
df_insure = df_insure.drop(columns=['Cause_of_Loss_Code'])
df_insure

In [ ]:
df_svi = df_svi[['FIPS','Crop','year','RPL_THEMES']]
df_cold = df_cold[['FIPS', 'year','Thr-2']]
df_drought = df_drought[['FIPS', 'year','pdsi-2']] # 'spi1y-2'
df_prcp = df_prcp[['FIPS', 'year','dpi1']]
df_vpd = df_vpd[['FIPS', 'year','VPD-Mean']]
df_warm = df_warm[['FIPS', 'year','Thr31']]


# **Failure Analysis**

In [ ]:
# df_fail , df_yield , df_svi , dfw , df_insure
df_fail

In [ ]:
cond_irig = df_fail['Irrigation Practice'] == 'ALL'
df_fail = df_fail[cond_irig]
df_fail['FIPS'] = df_fail['FIPS'].astype(int)

In [ ]:
df_merge_fail_ins = pd.merge(df_fail,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_fail_ins_svi = pd.merge(df_merge_fail_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_fail_ins_svi_gwu = pd.merge(df_merge_fail_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')
# df_merge_crop_ins_svi_gwu

df_merge_fail_ins_svi_gwu = df_merge_fail_ins_svi_gwu.dropna(subset='fail_share')
condy = df_merge_fail_ins_svi_gwu.year < 2023
df_merge_fail_ins_svi_gwu = df_merge_fail_ins_svi_gwu[condy]

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])
# df_merge_clim

df_merge = pd.merge(df_merge_fail_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()
df_merge

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]

df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]

In [ ]:
Cause_of_Loss[-3]

'Drought'

In [ ]:
# cond_ir = df_merge['Irrigation Practice'] == 'ALL'
# df_merge_fail = df_merge[cond_ir]
# df_merge_fail.to_csv('<DATA_ROOT>/Final_Exports_2024_2/failure_vars.csv')

In [ ]:
cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond_ir = df_merge['Irrigation Practice'] == 'ALL'
df_merge_fail = df_merge[cond_ir & cond_loss]

In [ ]:
df_merge_fail.groupby('Crop').count()

,FIPS,Irrigation Practice,year,fail_share,Cause_of_Loss_Desc,RPL_THEMES,gwu_rate,swu_rate,Thr-2,pdsi-2,dpi1,VPD-Mean,Thr31
Crop,,,,,,,,,,,,,
BARLEY,1281,1281,1281,1281,1281,1281,1281,1281,1281,1281,1281,1281,1281
CORN,16375,16375,16375,16375,16375,16375,16375,16375,16375,16375,16375,16375,16375
COTTON,3314,3314,3314,3314,3314,3314,3314,3314,3314,3314,3314,3314,3314
OATS,1894,1894,1894,1894,1894,1894,1894,1894,1894,1894,1894,1894,1894
SORGHUM,4709,4709,4709,4709,4709,4709,4709,4709,4709,4709,4709,4709,4709
SOYBEANS,14515,14515,14515,14515,14515,14515,14515,14515,14515,14515,14515,14515,14515
WHEAT,9662,9662,9662,9662,9662,9662,9662,9662,9662,9662,9662,9662,9662


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'CORN' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_fail.Crop == selected_crop
df = df_merge_fail[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# df_scaled

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_scaled.drop(columns=['fail_share'])
y = df_scaled['fail_share']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=85) #

params = {'max_depth': 20,
          'learning_rate': 0.01,
          'subsample': 0.6,
          'n_estimators': 750,
          }

xgb_model = xgb.XGBRegressor(**params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)


NameError: name 'df_scaled' is not defined